In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

def analyze_ab_test(events: pd.DataFrame, alpha: float = 0.05) -> dict:
    """
    Given raw ad event logs, compute CTR per group and run a z-test.

    Parameters:
        events : DataFrame with columns:
                   user_id    : str
                   group      : 'control' | 'treatment'
                   event_type : 'impression' | 'click'
        alpha  : significance level

    Returns dict with:
        ctr_control, ctr_treatment, lift_abs, lift_rel,
        z_stat, p_value, reject_null, conclusion
    """
    
    assert 'group'      in events.columns, "missing 'group' column"
    assert 'event_type' in events.columns, "missing 'event_type' column"
    assert 'user_id'    in events.columns, "missing 'user_id' column"
    assert 0 < alpha < 1

    aggregated = events.groupby(['group', 'event_type'])['user_id'].count()
    
    control_impressions = aggregated.loc[('control', 'impression')]
    control_clicks = aggregated.loc[('control', 'click')]
    treatment_impressions = aggregated.loc[('treatment', 'impression')]
    treatment_clicks = aggregated.loc[('treatment', 'click')]

    ctr_control = control_clicks / control_impressions
    ctr_treatment = treatment_clicks / treatment_impressions
    lift_abs = ctr_treatment - ctr_control
    lift_rel = lift_abs / ctr_control

    pooled = (control_clicks + treatment_clicks) / (control_impressions + treatment_impressions)
    se = np.sqrt(pooled * (1 - pooled) * (1 / control_impressions + 1 / treatment_impressions))
    z_stat = lift_abs / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

    reject_null = p_value < alpha

    if reject_null and lift_abs > 0:
        conclusion = 'Treatment significantly better'
    elif reject_null and lift_abs < 0:
        conclusion = 'Treatment significantly worse'
    else:
        conclusion = 'No significant difference'

    return {
        'ctr_control'  : ctr_control,
        'ctr_treatment': ctr_treatment,
        'lift_abs'     : lift_abs,
        'lift_rel'     : lift_rel,
        'z_stat'       : z_stat,
        'p_value'      : p_value,
        'reject_null'  : reject_null,
        'conclusion'   : conclusion
    }




In [7]:
# ── Test data ─────────────────────────────────────────────
np.random.seed(42)
n = 2000
events = pd.DataFrame({
    'user_id'   : [f'u{i}' for i in range(n)],
    'group'     : ['control'] * 1000 + ['treatment'] * 1000,
    'event_type': (
        np.random.choice(['impression','click'], 1000, p=[0.90, 0.10]).tolist() +
        np.random.choice(['impression','click'], 1000, p=[0.85, 0.15]).tolist()
    )
})

result = analyze_ab_test(events)
assert 0 < result['ctr_control']   < 1
assert 0 < result['ctr_treatment'] < 1
assert 0 <= result['p_value']      <= 1

print("Q1 passed ✅")

Q1 passed ✅
